# 📦 Project 07: Multi-Store Inventory Demand Forecasting with Stockout Risk
### Time Series Feature Engineering, LightGBM & Dynamic Safety Stock Optimization

**Author:** Data Science Portfolio Team  
**Difficulty:** 🟡 Intermediate  
**Domain:** Supply Chain & Logistics  

---
### Notebook Outline:
1. **Environment Setup & Imports**
2. **Data Ingestion & Multi-Store Time Series Inspection**
3. **Temporal EDA: Seasonality, Day-of-Week & Promotional Spikes**
4. **Lag & Rolling Window Feature Engineering (Preventing Leakage)**
5. **TimeSeriesSplit Cross-Validation**
6. **Model Benchmarks: Naive Lag vs. LightGBM Regressor**
7. **Dynamic Safety Stock & Stockout Risk Simulation**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Logistics forecasting environment ready.")

In [ ]:
# Ingestion & Date Parsing
df = pd.read_csv("data/warehouse_demand_timeseries.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['warehouse_id', 'sku_id', 'date']).reset_index(drop=True)
print(f"Time Series Records: {len(df)}")
display(df.head(4))

In [ ]:
# Visualizing Demand Curves across Warehouses
plt.figure(figsize=(14, 5))
sample_sku = "SKU-ELECTRONICS-01"
subset = df[df['sku_id'] == sample_sku]
sns.lineplot(data=subset, x='date', y='daily_sales', hue='warehouse_id', alpha=0.8)
plt.title(f"Daily Demand Trajectory for {sample_sku}", fontweight='bold')
plt.ylabel("Units Sold")
plt.show()

In [ ]:
# Feature Engineering: Lags & Rolling Window Statistics (Grouped by SKU & Warehouse)
for lag in [1, 7, 14]:
    df[f'lag_{lag}'] = df.groupby(['warehouse_id', 'sku_id'])['daily_sales'].shift(lag)

df['rolling_mean_7'] = df.groupby(['warehouse_id', 'sku_id'])['daily_sales'].transform(lambda x: x.shift(1).rolling(7).mean())
df['rolling_std_7'] = df.groupby(['warehouse_id', 'sku_id'])['daily_sales'].transform(lambda x: x.shift(1).rolling(7).std())

# Drop initial NaN rows created by shifting
df_clean = df.dropna().reset_index(drop=True)

feature_cols = ['is_promotion', 'day_of_week', 'month', 'lead_time_days',
                'lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_std_7']
target_col = 'daily_sales'

print(f"Cleaned feature matrix: {df_clean.shape}")

In [ ]:
# Time-Series Split Validation: Naive vs LightGBM
tscv = TimeSeriesSplit(n_splits=4)
X = df_clean[feature_cols]
y = df_clean[target_col]

wape_naive = []
wape_lgb = []

for train_idx, val_idx in tscv.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Naive baseline: lag_7
    naive_pred = X_val['lag_7']
    wape_naive.append(np.sum(np.abs(y_val - naive_pred)) / np.sum(y_val))
    
    # LightGBM Regressor
    model = lgb.LGBMRegressor(n_estimators=120, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1)
    model.fit(X_tr, y_tr)
    lgb_pred = model.predict(X_val)
    wape_lgb.append(np.sum(np.abs(y_val - lgb_pred)) / np.sum(y_val))

print(f"Mean WAPE - Naive Baseline: {np.mean(wape_naive):.2%}")
print(f"Mean WAPE - LightGBM:       {np.mean(wape_lgb):.2%}")

In [ ]:
# Dynamic Safety Stock Calculation (98% Service Level, Z=2.054)
Z = 2.054
df_clean['forecast_error'] = y - model.predict(X)
sigma_demand = df_clean.groupby(['warehouse_id', 'sku_id'])['forecast_error'].std().reset_index()
sigma_demand.rename(columns={'forecast_error': 'sigma_e'}, inplace=True)

# Merge lead times
lead_times = df_clean[['warehouse_id', 'sku_id', 'lead_time_days']].drop_duplicates()
safety_stock_df = pd.merge(sigma_demand, lead_times, on=['warehouse_id', 'sku_id'])
safety_stock_df['Safety_Stock_Units'] = np.ceil(Z * safety_stock_df['sigma_e'] * np.sqrt(safety_stock_df['lead_time_days'])).astype(int)

print("=== Recommended Warehouse Safety Stock Buffer (98% Service Level) ===")
display(safety_stock_df[['warehouse_id', 'sku_id', 'lead_time_days', 'Safety_Stock_Units']])